# AGENTS026 – Hours 5–6: Remediation Planning & Execution Agents

This notebook extends the RCA step into action. It creates:
- a `RemediationPlanningAgent` that converts an `RCAResult` plus policy rules into an ordered list of `Action` objects,
- an `ActionExecutorAgent` with mock infrastructure functions,
- and an orchestrator `handle_incident(incident_id)` that ties RCA, planning, and execution together.

The emphasis here is safe automation: some actions may be auto-executable, while others require approval based on policy.


## Section 1 – Install / Import Dependencies

We continue using PydanticAI with the local vLLM OpenAI-compatible endpoint.


In [1]:
%pip install -q pydantic-ai-slim openai pandas pyarrow

print('Installed / ensured pydantic-ai-slim, openai, pandas, pyarrow')


Note: you may need to restart the kernel to use updated packages.
Installed / ensured pydantic-ai-slim, openai, pandas, pyarrow


In [1]:
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any
import os
import json
import asyncio
import uuid

import pandas as pd
from pydantic import BaseModel, Field

from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

print('Imports OK')


Imports OK


## Section 2 – Load Prior Outputs

We load incident candidates and RCA results created in the earlier notebooks.


In [2]:
root = Path.cwd() / 'agents026'
data_dir = root / 'data'
incidents_dir = data_dir / 'incidents'
exec_dir = data_dir / 'execution'
exec_dir.mkdir(parents=True, exist_ok=True)

incident_candidates_path = incidents_dir / 'incident_candidates.json'
rca_results_path = incidents_dir / 'rca_results.json'

with open(incident_candidates_path, 'r', encoding='utf-8') as f:
    incident_candidates_raw = json.load(f)

with open(rca_results_path, 'r', encoding='utf-8') as f:
    rca_results_raw = json.load(f)

len(incident_candidates_raw), len(rca_results_raw)


(14, 3)

## Section 3 – Recreate Schemas

Define the data models used by planning and execution.


In [3]:
class IncidentCandidate(BaseModel):
    incident_id: str
    start_time: datetime
    end_time: datetime
    services: List[str]
    anomaly_type: str
    metric_summary: Dict[str, float] = Field(default_factory=dict)
    log_samples: List[str] = Field(default_factory=list)
    k8s_event_samples: List[str] = Field(default_factory=list)
    change_refs: List[str] = Field(default_factory=list)

class RCAResult(BaseModel):
    incident_id: str
    root_cause_hypothesis: str
    impacted_components: List[str]
    probable_trigger: Optional[str] = None
    evidence: List[str] = Field(default_factory=list)
    confidence: float = Field(..., ge=0.0, le=1.0)

class Action(BaseModel):
    action_id: str = Field(default_factory=lambda: f'act-{uuid.uuid4().hex[:8]}')
    action_type: str
    target: str
    description: str
    parameters: Dict[str, object] = Field(default_factory=dict)
    requires_approval: bool = True
    status: str = 'pending'
    expected_impact: Optional[str] = None
    rationale: Optional[str] = None

class ActionPlan(BaseModel):
    incident_id: str
    actions: List[Action]

class ExecutionResult(BaseModel):
    incident_id: str
    executed_actions: List[Action]
    skipped_actions: List[Action]
    notes: List[str] = Field(default_factory=list)

incident_candidates = [IncidentCandidate(**x) for x in incident_candidates_raw]
incident_map = {x.incident_id: x for x in incident_candidates}
rca_results = [RCAResult(**x) for x in rca_results_raw]
rca_map = {x.incident_id: x for x in rca_results}

list(rca_map.keys())[:5]


['inc-001', 'inc-014']

## Section 4 – Client Configuration for vLLM

Use the same local OpenAI-compatible vLLM endpoint as the previous notebook.


In [4]:
BASE_URL = os.environ.get('BASE_URL', 'http://localhost:8000/v1')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'abc-123')
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen3-30B-A3B')

provider = OpenAIProvider(base_url=BASE_URL, api_key=OPENAI_API_KEY)
chat_model = OpenAIChatModel(MODEL_NAME, provider=provider)

print({'BASE_URL': BASE_URL, 'MODEL_NAME': MODEL_NAME})


{'BASE_URL': 'http://localhost:8000/v1', 'MODEL_NAME': 'Qwen3-30B-A3B'}


## Section 5 – Policy Rules

These simple policy rules determine whether an action can be auto-executed or requires approval. You can tune them later for the demo.


In [5]:
POLICY_RULES = {
    'auto_approve_action_types': [
        'restart_service',
        'scale_service',
        'clear_cache',
    ],
    'approval_required_action_types': [
        'rollback_deploy',
        'rollback_config',
        'disable_feature_flag',
        'drain_node',
    ],
    'max_auto_scale_replicas': 2,
    'min_confidence_for_aggressive_action': 0.75,
}

POLICY_RULES


{'auto_approve_action_types': ['restart_service',
  'scale_service',
  'clear_cache'],
 'approval_required_action_types': ['rollback_deploy',
  'rollback_config',
  'disable_feature_flag',
  'drain_node'],
 'max_auto_scale_replicas': 2,
 'min_confidence_for_aggressive_action': 0.75}

## Section 6 – RemediationPlanningAgent

This agent converts an RCA result plus policy rules into an ordered `ActionPlan`.

We ask it to:
- keep actions operationally plausible,
- order them from safest to strongest,
- and mark approval-sensitive actions correctly.


In [6]:
class PlanningDeps(BaseModel):
    incident_map: Dict[str, IncidentCandidate]
    rca_map: Dict[str, RCAResult]
    policy_rules: Dict[str, object]

planning_deps = PlanningDeps(incident_map=incident_map, rca_map=rca_map, policy_rules=POLICY_RULES)

PLANNING_SYSTEM_PROMPT = '''
You are a remediation planning agent for production incidents.
Given an RCA result and policy rules, create a safe, ordered list of remediation actions.
Rules:
1. Start with the least risky action that can reduce blast radius or gather safety.
2. If rollback or config reversal is likely needed, mark it as requires_approval unless policy clearly allows automatic execution.
3. Use concrete action types such as restart_service, scale_service, rollback_deploy, rollback_config, clear_cache, disable_feature_flag.
4. Every action must include target, description, parameters, expected_impact, rationale, and requires_approval.
5. Return ActionPlan only.
'''

planning_agent = Agent(
    model=chat_model,
    deps_type=PlanningDeps,
    output_type=ActionPlan,
    system_prompt=PLANNING_SYSTEM_PROMPT,
)

@planning_agent.tool
def get_rca_result(ctx: RunContext[PlanningDeps], incident_id: str) -> Dict[str, object]:
    return ctx.deps.rca_map[incident_id].model_dump()

@planning_agent.tool
def get_policy_rules(ctx: RunContext[PlanningDeps]) -> Dict[str, object]:
    return ctx.deps.policy_rules

@planning_agent.tool
def get_incident_context(ctx: RunContext[PlanningDeps], incident_id: str) -> Dict[str, object]:
    return ctx.deps.incident_map[incident_id].model_dump()

print('RemediationPlanningAgent ready')


RemediationPlanningAgent ready


In [7]:
class RemediationPlanningAgentWrapper:
    def __init__(self, agent: Agent, deps: PlanningDeps):
        self.agent = agent
        self.deps = deps

    def build_prompt(self, incident_id: str) -> str:
        return (
            f'Build a remediation plan for incident {incident_id}. '
            f'Use RCA result, policy rules, and incident context. Return ordered ActionPlan.'
        )

    async def plan_async(self, incident_id: str) -> ActionPlan:
        result = await self.agent.run(self.build_prompt(incident_id), deps=self.deps)
        return result.output

    def plan(self, incident_id: str) -> ActionPlan:
        return asyncio.run(self.plan_async(incident_id))

RemediationPlanningAgent = RemediationPlanningAgentWrapper(planning_agent, planning_deps)


## Section 7 – Mock Infrastructure Functions

These functions simulate infra actions. In the final demo, they provide clear, auditable execution logs without touching real systems.


In [8]:
mock_execution_log = []

def mock_restart_service(service: str) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated restart of service {service}'}
    mock_execution_log.append({'action': 'restart_service', 'target': service, **result})
    return result

def mock_scale_service(service: str, replicas_delta: int = 1) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated scale of {service} by +{replicas_delta} replicas'}
    mock_execution_log.append({'action': 'scale_service', 'target': service, 'replicas_delta': replicas_delta, **result})
    return result

def mock_clear_cache(service: str) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated cache clear for {service}'}
    mock_execution_log.append({'action': 'clear_cache', 'target': service, **result})
    return result

def mock_rollback_deploy(service: str, version: Optional[str] = None) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated rollback for {service} to previous stable version {version or "auto"}'}
    mock_execution_log.append({'action': 'rollback_deploy', 'target': service, 'version': version, **result})
    return result

def mock_rollback_config(service: str) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated config rollback for {service}'}
    mock_execution_log.append({'action': 'rollback_config', 'target': service, **result})
    return result

def mock_disable_feature_flag(service: str, flag_name: str = 'unknown_flag') -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated disabling feature flag {flag_name} for {service}'}
    mock_execution_log.append({'action': 'disable_feature_flag', 'target': service, 'flag_name': flag_name, **result})
    return result


## Section 8 – ActionExecutorAgent

This agent decides whether actions are executable under policy and routes them to the appropriate mock infra functions.

For demo safety, the actual infrastructure mutation is mocked, but the control flow is realistic.


In [9]:
class ExecutorDeps(BaseModel):
    policy_rules: Dict[str, object]

executor_deps = ExecutorDeps(policy_rules=POLICY_RULES)

EXECUTION_SYSTEM_PROMPT = '''
You are an action execution agent.
Evaluate each proposed action against policy.
Only execute actions that are safe and do not require approval.
If an action requires approval, leave it pending/skipped and explain why.
You do not invent new actions; you evaluate and route the given actions.
'''

executor_agent = Agent(
    model=chat_model,
    deps_type=ExecutorDeps,
    output_type=ExecutionResult,
    system_prompt=EXECUTION_SYSTEM_PROMPT,
)

@executor_agent.tool
def get_execution_policy(ctx: RunContext[ExecutorDeps]) -> Dict[str, object]:
    return ctx.deps.policy_rules

print('ActionExecutorAgent base ready')


ActionExecutorAgent base ready


In [10]:
class ActionExecutorAgentWrapper:
    def __init__(self, agent: Agent, deps: ExecutorDeps):
        self.agent = agent
        self.deps = deps

    def execute_action(self, action: Action) -> Action:
        if action.requires_approval:
            action.status = 'skipped'
            return action

        if action.action_type == 'restart_service':
            result = mock_restart_service(action.target)
            action.status = result['status']
        elif action.action_type == 'scale_service':
            replicas_delta = int(action.parameters.get('replicas_delta', 1))
            result = mock_scale_service(action.target, replicas_delta=replicas_delta)
            action.status = result['status']
        elif action.action_type == 'clear_cache':
            result = mock_clear_cache(action.target)
            action.status = result['status']
        elif action.action_type == 'rollback_deploy':
            version = action.parameters.get('version')
            result = mock_rollback_deploy(action.target, version=version)
            action.status = result['status']
        elif action.action_type == 'rollback_config':
            result = mock_rollback_config(action.target)
            action.status = result['status']
        elif action.action_type == 'disable_feature_flag':
            flag_name = str(action.parameters.get('flag_name', 'unknown_flag'))
            result = mock_disable_feature_flag(action.target, flag_name=flag_name)
            action.status = result['status']
        else:
            action.status = 'skipped'
        return action

    async def evaluate_and_execute_async(self, incident_id: str, actions: List[Action]) -> ExecutionResult:
        executed_actions = []
        skipped_actions = []
        notes = []

        for action in actions:
            if action.requires_approval:
                action.status = 'skipped'
                skipped_actions.append(action)
                notes.append(f'{action.action_id}: approval required for {action.action_type}')
                continue
            updated = self.execute_action(action)
            if updated.status == 'executed':
                executed_actions.append(updated)
            else:
                skipped_actions.append(updated)
                notes.append(f'{updated.action_id}: action not executed ({updated.action_type})')

        return ExecutionResult(
            incident_id=incident_id,
            executed_actions=executed_actions,
            skipped_actions=skipped_actions,
            notes=notes,
        )

    def evaluate_and_execute(self, incident_id: str, actions: List[Action]) -> ExecutionResult:
        return asyncio.run(self.evaluate_and_execute_async(incident_id, actions))

ActionExecutorAgent = ActionExecutorAgentWrapper(executor_agent, executor_deps)
print('ActionExecutorAgent wrapper initialized')


ActionExecutorAgent wrapper initialized


## Section 9 – Orchestrator: handle_incident(incident_id)

This function is the first full end-to-end orchestration step in the workflow.

Flow:
1. Retrieve RCA result for the incident.
2. Ask `RemediationPlanningAgent` for an ordered action plan.
3. Pass the plan to `ActionExecutorAgent`.
4. Persist outputs for later reporting/demo.


In [11]:
async def handle_incident(incident_id: str) -> Dict[str, object]:
    if incident_id not in rca_map:
        raise ValueError(f'No RCA result found for incident_id={incident_id}')

    plan = await RemediationPlanningAgent.plan_async(incident_id)
    execution = await ActionExecutorAgent.evaluate_and_execute_async(incident_id, plan.actions)

    bundle = {
        'incident_id': incident_id,
        'rca_result': rca_map[incident_id].model_dump(),
        'action_plan': plan.model_dump(),
        'execution_result': execution.model_dump(),
    }

    out_path = exec_dir / f'{incident_id}_workflow.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(bundle, f, default=str, indent=2)

    return bundle


## Section 10 – Test Planning Agent

First verify the remediation plan alone on one sample incident.


In [12]:
sample_incident_ids = list(rca_map.keys())[:2]
sample_incident_ids


['inc-001', 'inc-014']

In [13]:
if sample_incident_ids:
    sample_plan = await RemediationPlanningAgent.plan_async(sample_incident_ids[0])
    print(sample_plan.model_dump_json(indent=2))
else:
    print('No RCA outputs available; run the previous RCA notebook first.')


{
  "incident_id": "inc-001",
  "actions": [
    {
      "action_id": "act-ae00267f",
      "action_type": "clear_cache",
      "target": "auth-service",
      "description": "Clear the cache for auth-service to mitigate potential issues caused by stale data.",
      "parameters": {},
      "requires_approval": false,
      "status": "pending",
      "expected_impact": "Low - May temporarily affect performance as cache is rebuilt.",
      "rationale": "The RCA suggests a configuration change might have led to stale data causing timeouts. Clearing the cache can help refresh the data and reduce error rates."
    },
    {
      "action_id": "act-9ac26c07",
      "action_type": "restart_service",
      "target": "auth-service",
      "description": "Restart the auth-service to apply configuration changes and resolve any transient issues.",
      "parameters": {},
      "requires_approval": false,
      "status": "pending",
      "expected_impact": "Medium - Service will be briefly unavaila

## Section 11 – Test End-to-End Orchestration

Now run `handle_incident(incident_id)` for one or two incidents and inspect both the action plan and the execution results.


In [14]:
workflow_outputs = []
for iid in sample_incident_ids[:2]:
    try:
        bundle = await handle_incident(iid)
        workflow_outputs.append(bundle)
        print('\n===== WORKFLOW OUTPUT FOR', iid, '=====')
        print(json.dumps(bundle, indent=2)[:6000])
    except Exception as e:
        print('Failed for', iid, ':', repr(e))



===== WORKFLOW OUTPUT FOR inc-001 =====
{
  "incident_id": "inc-001",
  "rca_result": {
    "incident_id": "inc-001",
    "root_cause_hypothesis": "The error rate spike in auth-service is likely caused by a recent configuration change applied at 2026-06-10T13:01:00.",
    "impacted_components": [
      "auth-service"
    ],
    "probable_trigger": "config_change",
    "evidence": [
      "Error rate increased to 0.1125 at 2026-06-10 13:14:00",
      "Log entry at 2026-06-10T13:14:00: 'auth-service: timeout while calling upstream service'",
      "Recent config change applied to auth-service at 2026-06-10T13:01:00"
    ],
    "confidence": 0.85
  },
  "action_plan": {
    "incident_id": "inc-001",
    "actions": [
      {
        "action_id": "restart-auth-service",
        "action_type": "restart_service",
        "target": "auth-service",
        "description": "Restart the auth-service to mitigate potential transient issues caused by the recent configuration change.",
        "param

## Section 12 – Execution Log Inspection

Review the mock infrastructure execution log. This is useful in the demo to show auditability and policy-aware automation.


In [15]:
pd.DataFrame(mock_execution_log) if mock_execution_log else pd.DataFrame(columns=['action','target','status','message'])


,action,target,status,message,replicas_delta
0,restart_service,auth-service,executed,Simulated restart of service auth-service,NaN
1,scale_service,auth-service,executed,Simulated scale of auth-service by +1 replicas,1.0
2,clear_cache,auth-service,executed,Simulated cache clear for auth-service,NaN


## Section 13 – Persist Summary Artifacts

Save a compact summary of workflow outputs so the next notebook can build reporting or a demo narrative from them.


In [16]:
summary_rows = []
for bundle in workflow_outputs:
    exec_result = bundle['execution_result']
    summary_rows.append({
        'incident_id': bundle['incident_id'],
        'planned_actions': len(bundle['action_plan']['actions']),
        'executed_actions': len(exec_result['executed_actions']),
        'skipped_actions': len(exec_result['skipped_actions']),
        'root_cause_hypothesis': bundle['rca_result']['root_cause_hypothesis'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = exec_dir / 'workflow_summary.parquet'
summary_df.to_parquet(summary_path, index=False)

print('Saved summary to:', summary_path)
summary_df


Saved summary to: /workspace/shared/agents026/data/execution/workflow_summary.parquet


,incident_id,planned_actions,executed_actions,skipped_actions,root_cause_hypothesis
0,inc-001,5,3,2,The error rate spike in auth-service is likely...
1,inc-014,5,0,5,The root cause is a recent deployment that int...


In [17]:
def enforce_policy(action: Action) -> Action:
    """Deterministic policy gate — overrides the LLM's requires_approval flag."""
    if action.action_type in POLICY_RULES['auto_approve_action_types']:
        action.requires_approval = False
    elif action.action_type in POLICY_RULES['approval_required_action_types']:
        action.requires_approval = True
    return action

# Re-run inc-014 execution with policy enforced
bundle14 = [b for b in workflow_outputs if b['incident_id'] == 'inc-014'][0]
plan_actions = [enforce_policy(Action(**a)) for a in bundle14['action_plan']['actions']]
for a in plan_actions:
    a.status = 'pending'

execution14 = await ActionExecutorAgent.evaluate_and_execute_async('inc-014', plan_actions)
bundle14['action_plan']['actions'] = [a.model_dump() for a in plan_actions]
bundle14['execution_result'] = execution14.model_dump()

with open(exec_dir / 'inc-014_workflow.json', 'w', encoding='utf-8') as f:
    json.dump(bundle14, f, default=str, indent=2)

print('Executed:', [a.action_type for a in execution14.executed_actions])
print('Skipped (pending approval):', [a.action_type for a in execution14.skipped_actions])

# Re-save summary
summary_rows = []
for bundle in workflow_outputs:
    er = bundle['execution_result']
    summary_rows.append({
        'incident_id': bundle['incident_id'],
        'planned_actions': len(bundle['action_plan']['actions']),
        'executed_actions': len(er['executed_actions']),
        'skipped_actions': len(er['skipped_actions']),
        'root_cause_hypothesis': bundle['rca_result']['root_cause_hypothesis'],
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_parquet(exec_dir / 'workflow_summary.parquet', index=False)
summary_df

Executed: ['restart_service', 'scale_service', 'clear_cache']
Skipped (pending approval): ['rollback_deploy', 'disable_feature_flag']


,incident_id,planned_actions,executed_actions,skipped_actions,root_cause_hypothesis
0,inc-001,5,3,2,The error rate spike in auth-service is likely...
1,inc-014,5,3,2,The root cause is a recent deployment that int...
